# Library Imports 

In [1]:
import pickle
import os
from pathlib import Path
from collections import defaultdict

from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample
import scipy.stats as stats
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pandas as pd
from sklearn.model_selection import GroupKFold
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from behave_analysis.utils.creating_directories import make_directory
from behave_analysis.visualize.visualize_utils import open_tracking_data
from behave_analysis.process.session import get_experiment
from behave_analysis.utils.arena_plotting import Arena

# Data Imports

In [2]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept
from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept
from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept
from behave_analysis.database.Experiments.JAL006_ex import JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, JAL6_flip7_1apr
from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr,JAL7_30apr
from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_tiny_3may, JAL8_flip4_10may, JAL8_14may, JAL8_21may, JAL8_flip3_7may

# Defined session names, experiments and hash mice to sessions

In [3]:
# The values must be in order of data for the code to work
mice_groups = {
    "JAL6": ['JAL6_flip3_18mar', 'JAL6_flip4_21mar', 'JAL6_flip5_25mar', 'JAL6_28mar'],
    "JAL3": ['JAL3_25aug', 'JAL3_1sept', 'JAL3_4sept', 'JAL3_7sept'],
    "JAL7": ['JAL7_flip2_12mar', 'JAL7_flip5_22mar', 'JAL7_sesh8_9apr', 'JAL7_sesh9_16apr', 'JAL7_23apr'],
    "JAL8": ['JAL8_flip1_25apr', 'JAL8_flip2_29apr', 'JAL8_flip4_10may', "JAL8_flip3_7may", 'JAL8_14may'],
    "JAL4": ['JAL4_28aug', 'JAL4_3rdSept', 'JAL4_11thSept', 'JAL4_19thSept'],
    "JAL5": ['JAL5_8thSept', 'JAL5_21stSept']}

# The experiment objects index must match the session name index for the code to work
experiments_objects = [JAL6_flip3_18mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_28mar,
                       JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept,
                       JAL005_8thSept, JAL005_21stSept,
                       JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr,
                       JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may, JAL8_flip3_7may,
                       JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept]

# The session name indexes must match the experiment object indexes
session_names = ["JAL6_flip3_18mar", "JAL6_flip4_21mar", "JAL6_flip5_25mar", "JAL6_28mar",
                 "JAL3_25aug", "JAL3_1sept", "JAL3_4sept", "JAL3_7sept",
                 "JAL5_8thSept", "JAL5_21stSept",
                 "JAL7_sesh8_9apr", "JAL7_sesh9_16apr", "JAL7_flip5_22mar", "JAL7_flip2_12mar", "JAL7_23apr",
                 "JAL8_flip1_25apr", "JAL8_flip2_29apr", "JAL8_flip4_10may", "JAL8_14may", "JAL8_flip3_7may",
                 "JAL4_3rdSept", "JAL4_19thSept", "JAL4_28aug", "JAL4_11thSept"]

# Initilisation 

In [4]:
dir = make_directory(r"Z:\Jasmine_Laurence\single_trial_overview\decoding_spatial_efficiency")
dir = Path(dir)
SE_THRESHOLD = 0.90

# Define some helper functions to use

Note that I have added frame numbers to aid the plotting of train and test distributions for the past design matrix function only, thus plotting will fail on other functions

In [5]:
# ------ Loading helper functions ------
def load(dir, file_name):
    """Ensure you include the .pkl in the file_name"""
    dir = dir / file_name
    with open(dir, "rb") as f:
        file = pickle.load(f)
    return file

def load_video_data(experiment):
    """Loads the video data for the experiment"""
    loaded_session = get_experiment(experiment)
    try:
        video_df = pl.read_csv(os.path.join(loaded_session.base_path, loaded_session.processed_path) + "\\" "full_video_dataframe.csv")

    except FileNotFoundError:
        print("One of the files was not found")
        return None
    
    return video_df

def extract_behavioural_data_between_homing_onset_offset(homing_object: dict, video_df: pl.DataFrame) -> list:
    """Returns a list of behavioural dataframes for each homing event in the homing object."""
    homing_info = []
    for onset, offset in zip(homing_object.onset_frames, homing_object.offset_frames):
        homing = video_df[int(onset) - 1 : int(offset)]  # Frames -1 because of 0 indexing, 
        homing = homing.select(
            [
                "frames",
                "mouse_x_position",
                "mouse_y_position",
                "hdir",
                "hsa",
                "h_preflipbar_a",
                "h_postflipbar_a",
            ]
        )
        homing_info.append(homing)
    return homing_info

def add_homing_id_to_homing_data(extracted_homing_info: list) -> list:
    """Adding the homing id (abitrary ascending interger) to the homing data.
    This is needed for the group cross validation object

    Args:
        extracted_homing_info (list): A list of homing dataframes for each homing period

    Returns:
        (list) of homing dataframes with the homing id added as a column
    """
    for idx, homing in enumerate(extracted_homing_info):
        updated_homing = homing.with_columns(pl.lit(idx).alias("homing_id"))
        extracted_homing_info[idx] = updated_homing
    return extracted_homing_info

def create_the_design_matrix(homing_data: pl.DataFrame, frame_by_cluster_matrix: np.ndarray, classes: list) -> np.ndarray:
    spike_data_per_homing = []
    classes_extended = []
    homing_ids = []
    for i, homing in enumerate(homing_data):
        first_frame = homing["frames"][0] - 1  # Frames -1 because of 0 indexing
        last_frame = homing["frames"][-1] # Do not minus 1 for last frame as it is not inclusive in the slicing
        homing_id = homing["homing_id"].to_numpy().reshape(-1, 1)
        homing_ids.append(homing_id)
        spike_data = frame_by_cluster_matrix[first_frame : last_frame]  
        spike_data_per_homing.append(spike_data)
        classes_extended.append([classes[i]] * (last_frame - first_frame))
        assert len(spike_data) == len(classes_extended[i])
    design_matrix = np.vstack(spike_data_per_homing) # vertically stack the spike data for each homing into a design matrix
    return design_matrix, classes_extended, homing_ids

def create_the_past_design_matrix(homing_data: pl.DataFrame, frame_by_cluster_matrix: np.ndarray, classes: list, shift: int = 20) -> np.ndarray:
    """Pull neural data from before the homing onset by a certain amount of frames
    
    ARgs:
        shift (int) the number of frames to shift the neural data by to look before the homing"""
    spike_data_per_homing = []
    classes_extended = []
    homing_ids = []
    frame_numbers = []
    for i, homing in enumerate(homing_data):
        first_frame = max(homing["frames"][0] - 1 - shift, 0)  # Ensure non-negative index
        last_frame = homing["frames"][0] - 1 # The last frame is the first frame of the homing, -1 because of 0 indexing frames start at 1
        
        # Extract homing IDs and flatten appropriately
        homing_ids.extend(homing["homing_id"][:last_frame - first_frame])  # Extend with flattened values
        
        # Extract the spike data
        spike_data = frame_by_cluster_matrix[first_frame : last_frame] 
        spike_data_per_homing.append(spike_data)
        
        classes_extended.append([classes[i]] * (last_frame - first_frame))
        assert len(spike_data) == len(classes_extended[i])
        frames = np.arange(first_frame, last_frame)
        frame_numbers.extend(frames)
    design_matrix = np.vstack(spike_data_per_homing) # vertically stack the spike data for each homing into a design matrix
    
    # Vertically append the frame numbers to the design matrix
    frame_numbers = np.array(frame_numbers).reshape(-1, 1)
    design_matrix = np.hstack((design_matrix, frame_numbers))
    
    return design_matrix, classes_extended, homing_ids

def create_the_random_design_matrix(homing_data: pl.DataFrame, frame_by_cluster_matrix: np.ndarray, classes: list, period=200):
    """
        Homing_data (List): Of dataframes containing frames, mouse pos, hdir, hsa, bar angles, homing_id
    """
    spike_data_per_homing = []
    classes_extended = []
    homing_ids = []
    M, N = frame_by_cluster_matrix.shape # M is number of frames in entire session
    frames = np.arange(0, M - period)  # Ensure range is valid
        
    for i, homing in enumerate(homing_data):
        first_frame = np.random.choice(frames)
        last_frame = first_frame + period
        
        # Validate slice bounds
        if last_frame > M:
            continue
        
        spike_data = frame_by_cluster_matrix[first_frame:last_frame]
        spike_data_per_homing.append(spike_data)
        
        # Append consistent homing IDs and classes
        id = homing["homing_id"][0]
        homing_ids.append([id] * period)
        classes_extended.append([classes[i]] * period)
    
    design_matrix = np.vstack(spike_data_per_homing)
    
    return design_matrix, np.array(classes_extended), np.array(homing_ids)

def create_histograms(homings_above_the_barrier, classes2):
    """Creates histograms for mouse_x_position, mouse_y_position, hdir, and hsa,
    comparing the two classes, with separate colors for each class."""
    
    metrics = ["mouse_x_position", "mouse_y_position", "hdir", "hsa"]
    colors = {0: 'blue', 1: 'red'}  # Define colors for classes
    
    for metric in metrics:
        class_0_data = []
        class_1_data = []

        for dataframe, cls in zip(homings_above_the_barrier, classes2):
            if cls == 0:
                class_0_data.extend(dataframe[metric].to_list())
            elif cls == 1:
                class_1_data.extend(dataframe[metric].to_list())

        plt.figure()
        plt.hist(class_0_data, bins=30, alpha=0.7, label="Class 0", color=colors[0])
        plt.hist(class_1_data, bins=30, alpha=0.7, label="Class 1", color=colors[1])
        plt.title(f"Histogram of {metric}")
        plt.xlabel(metric)
        plt.ylabel("Frequency")
        plt.legend()
        plt.show()


# Plotting functions

In [6]:
def control_for_500ms_before_homing(experiments_objects, session_names, dir):
    
    # Initialize distributions for Class 0 and Class 1
    class_0_dist = []
    class_1_dist = []

    # First get homing onsets across all sessions
    for experiment, session_name in zip(experiments_objects, session_names):
        print(f"Running 500ms control before homing for session: {session_name}")
        loaded_session = get_experiment(experiment)

        # Set paths
        base_path = loaded_session.base_path
        processed_path = loaded_session.processed_path
        session_path = os.path.join(base_path, processed_path)
        homing_path = os.path.join(session_path, "homings", "homings_obj.pkl")
        
        # Load homing onset and video data
        try:
            video_df = pd.read_csv(os.path.join(loaded_session.base_path, loaded_session.processed_path, "full_video_dataframe.csv"))
            with open(homing_path, "rb") as hf:
                homings_object = pickle.load(hf)
        except FileNotFoundError:
            print(f"Skipping session {session_name} because homing or video data is missing")
            continue

        # Extract homing data
        onsets = homings_object.onset_frames  # list of onset frames
        homing_class = [1 if seff > SE_THRESHOLD else 0 for seff in homings_object.spatial_efficiency]  # List of classes

        # Extract behavioral data for each homing event
        for onset, cls in zip(onsets, homing_class):
            first_frame = onset - 20  # 500ms before homing onset
            behaviour = video_df.iloc[first_frame:onset]

            # Append X positions to the respective class distribution
            if cls == 1:
                class_1_dist.extend(behaviour["mouse_x_position"].values)
            else:
                class_0_dist.extend(behaviour["mouse_x_position"].values)

        # Plot the distributions
        plt.figure(figsize=(10, 6))
        plt.hist(class_0_dist, bins=30, alpha=0.7, color="blue", label="Class 0")
        plt.hist(class_1_dist, bins=30, alpha=0.7, color="red", label="Class 1")
        plt.title("Comparison of Mouse X Position Distributions Before Down Sampling for Class 0 and Class 1 500ms Before Homing")
        plt.suptitle(f"Session: {session_name}")
        plt.xlabel("Position")
        plt.ylabel("Frequency")
        plt.legend()
        
        # Dir
        save_path = dir / "500ms_before_control"
        make_directory(save_path)
        plt.savefig(save_path / f"{session_name}.png")
        plt.close()

def good_vs_bad_trajectories_plotted_to_arena(barrier_location, tracking_data, homings_above_the_barrier, classes, homing_conditions, dir, session_name):
    fig, (ax_pre_flip, ax_post_flip) = plt.subplots(1, 2, figsize=(20, 16))
    Arena(ax=ax_pre_flip,
        shelter_coordinates=tracking_data["shelter_loc"],
        condition="barrier_pre_flip",
        barrier_coordinates=barrier_location)
    Arena(ax=ax_post_flip,
        shelter_coordinates=tracking_data["shelter_loc"],
        condition="barrier_post_flip",
        barrier_coordinates=barrier_location)
    for i, homing in enumerate(homings_above_the_barrier):
        if homing_conditions[i] == "barrier_pre_flip":
            if classes[i] == 0:
                ax_pre_flip.plot(homing["mouse_x_position"], homing["mouse_y_position"], color="r", label = "Bad")                    
            if classes[i] == 1:
                ax_pre_flip.plot(homing["mouse_x_position"], homing["mouse_y_position"], color="g", label = "Good")
        elif homing_conditions[i] == "barrier_post_flip":
            if classes[i] == 0:
                ax_post_flip.plot(homing["mouse_x_position"], homing["mouse_y_position"], color="r", label = "Bad")
            if classes[i] == 1:
                ax_post_flip.plot(homing["mouse_x_position"], homing["mouse_y_position"], color="g", label = "Good")
    
    handles, labels = plt.gca().get_legend_handles_labels()
    unique_handles_labels = dict(zip(labels, handles))
    ax_pre_flip.legend(unique_handles_labels.values(), unique_handles_labels.keys())
    ax_post_flip.legend(unique_handles_labels.values(), unique_handles_labels.keys())
    ax_pre_flip.set_title("Pre Flip Barrier")
    ax_post_flip.set_title("Post Flip Barrier")
    fig.suptitle(f"Good vs Bad Trajectories for {session_name}")
    
    # Dir
    save_path = dir / "good_vs_bad_trajectories"
    make_directory(save_path)
    plt.savefig(save_path / f"{session_name}.png")
    plt.close()


# Main functions

In [7]:
def produce_data(experiments_objects, session_names, name_of_storage, create_design_matrix):
    storage = defaultdict(defaultdict)
    for experiment, session_name in zip(experiments_objects, session_names):
        print(f"Loading data for session: {session_name}")
        loaded_session = get_experiment(experiment)

        # Set paths
        base_path = loaded_session.base_path
        processed_path = loaded_session.processed_path
        session_path = os.path.join(base_path, processed_path)
        homing_path = os.path.join(session_path, "homings", "homings_obj.pkl")
        
        # Load data
        try:
            video_df = pl.read_csv(os.path.join(loaded_session.base_path, loaded_session.processed_path) + "\\" "full_video_dataframe.csv")
            with open(homing_path, "rb") as hf:
                    homings_object = pickle.load(hf) 
                    
            frame_by_cluster_matrix = np.load(os.path.join(loaded_session.base_path, loaded_session.processed_path) + "\\" + "frame_by_" + "good" + "_cluster_matrix.npy")
            tracking_data = open_tracking_data(loaded_session)

        except FileNotFoundError:
            print("One of the files was not found")

        #     tracking_data["barrier_loc"] = [[224, 515], [797, 512], [510, 513]]
            
        # Extract logic data
        barrier_location = tracking_data["barrier_loc"]
        homing_info = extract_behavioural_data_between_homing_onset_offset(homings_object, video_df)
        homing_info = add_homing_id_to_homing_data(homing_info)
        homing_conditions = homings_object.homing_condition
        homing_class = [1 if seff > SE_THRESHOLD else 0 for seff in homings_object.spatial_efficiency]
        
        # Only keep the homings where the barrier is present
        barrier_homings = [i for i, homing in enumerate(homing_conditions) if homing in ["barrier_pre_flip", "barrier_post_flip"]]
        classes1 = [homing_class[i] for i in barrier_homings]
        homing_info1 = [homing_info[i] for i in barrier_homings]
        homing_conditions1 = [homing_conditions[i] for i in barrier_homings]
        
        # Only keep the homings above the barrier
        homings_above_the_barrier = [homing for i, homing in enumerate(homing_info1) if homing["mouse_y_position"][0] < barrier_location[0][1]]
        classes2 = [classes1[i] for i, homing in enumerate(homing_info1) if homing["mouse_y_position"][0] < barrier_location[0][1]]
        homing_conditions2 = [homing_conditions1[i] for i, homing in enumerate(homing_info1) if homing["mouse_y_position"][0] < barrier_location[0][1]]
        
        # Plot the trajectories of good and bad homings
        good_vs_bad_trajectories_plotted_to_arena(barrier_location, tracking_data, homings_above_the_barrier, classes2, homing_conditions2, dir, session_name)
        
        # Create design matrix
        design_matrix, classes_extended, homing_ids = create_design_matrix(homings_above_the_barrier, frame_by_cluster_matrix, classes2)
        classes_extended = [item for sublist in classes_extended for item in sublist]  # Flatten the classes_extended list

        # Store the data
        storage[session_name]["design_matrix"] = design_matrix
        storage[session_name]["classes_extended"] = classes_extended
        storage[session_name]["homing_ids"] = homing_ids
        
    # Save the data as pickle
    with open(dir / f"{name_of_storage}.pkl", "wb") as f:
        pickle.dump(storage, f)
    
    return storage

Functions to support the actual model

In [11]:
def count_class_labels(classes):
    dic_to_store = defaultdict(int)
    for idx, val in enumerate(classes):
        dic_to_store[val] += 1
    return dic_to_store

def downsample_larger_class(design_matrix, classes_extended, random_state, homing_ids):
    'Randomly downsamples the larger class to match the smaller class'
    X = design_matrix
    y = classes_extended
    class_0_indices = np.where(y == 0)[0]
    class_1_indices = np.where(y == 1)[0]

    # Ensure only downsampling of the larger class
    if len(class_0_indices) > len(class_1_indices):
        class_0_indices = resample(class_0_indices, replace=False, n_samples=len(class_1_indices), random_state=random_state)
    elif len(class_1_indices) > len(class_0_indices):
        class_1_indices = resample(class_1_indices, replace=False, n_samples=len(class_0_indices), random_state=random_state)

    balanced_indices = np.concatenate([class_0_indices, class_1_indices])
    X_balanced = X[balanced_indices]
    y_balanced = y[balanced_indices]
    print("The length of balanced indcies", len(balanced_indices))
    print("The length of homing_ids", len(homing_ids))
    #groups_balanced = homing_ids[balanced_indices]
    groups_balanced = [homing_ids[i] for i in balanced_indices]


    return X_balanced, y_balanced, groups_balanced

def check_there_is_enough_data(count_ans, session_name, cutoff):
    """Returns True if there is enough data
    
    Args:
        cutoff (int): The minimum number of frames required for each class. 40 frames is 1 second of data"""
    if count_ans[0] < cutoff or count_ans[1] < cutoff:
        print(f"For session {session_name}, there is less than {cutoff / 40} seconds of data for one class, skipping session")
        return False
    return True

def check_there_are_two_classes(y_train):
    if len(np.unique(y_train)) == 1:
        print("Skipping fold because there is only one class")
        return False
    return True

def handle_class_imbalance(classes_extended, session_name, design_matrix, data):
    # Check the class distribution before down sampling
    count_ans = count_class_labels(classes_extended) # How many frames have 0 or 1
    print(f"Class distribution before down sampling. 0: {count_ans[0]}, 1: {count_ans[1]}")
    if not check_there_is_enough_data(count_ans, session_name, cutoff=80):  # If there are less than x seconds of data in either class, skip this session
        return None, None, None # The cuttoff has not been met

    # Down sample the larger class
    X_balanced, y_balanced, groups_balanced = downsample_larger_class(design_matrix=design_matrix, classes_extended=classes_extended, random_state=1337, homing_ids=data["homing_ids"])
    count_of_downsampled_classes = count_class_labels(y_balanced)
    print(f"Class distribution for the entire session after down sampling. 0: {count_of_downsampled_classes[0]}, 1: {count_of_downsampled_classes[1]}")
    
    return X_balanced, y_balanced, groups_balanced

def are_the_distributions_different(good_data, bad_data):
    """Conduct a ks_statistic test to see if the distributions are different. If p < 0.05, then the distributions are different"""
    ks_statistic, p_value = stats.ks_2samp(good_data, bad_data)
    if p_value < 0.05:
        return True, ks_statistic, p_value
    return False, ks_statistic, p_value

def compute_accuracy_data(data_type, data_type_name, random_labels = False):
    
    # Save files
    path = os.path.join("Z:\Jasmine_Laurence\single_trial_overview\decoding_spatial_efficiency", data_type_name)
    dir = make_directory(path)
          
    # Create a dictionary to store the accuracy for each session
    whole_homing = defaultdict(list)
    ks_decoding_comparison = defaultdict(defaultdict)
    coefs = defaultdict(defaultdict)

    # Loop through each session and run logistic regression
    for (session_name, data), experiment in zip(data_type.items(), experiments_objects):
        print(f"Running logistic regression for session: {session_name}")
     
        # Load required data
        video_df = load_video_data(experiment)
        design_matrix = data["design_matrix"] # The X data 
        classes_extended = np.asarray(data["classes_extended"])  # The y data        
        
        if random_labels:
            np.random.shuffle(classes_extended) # randomly shuffle the classes

        # Down sample the larger class
        X_balanced, y_balanced, groups_balanced = handle_class_imbalance(classes_extended, session_name, design_matrix, data)
        if X_balanced is None: # If the cuttoff for the amount of data has not been met
            continue # Skip this session
        
        group_kfold = GroupKFold(n_splits=2) # k-fold interator variation with non-overlapping groups
        fold_accuracies = []
        
        # Assign and remove the last column of the design matrix - Frame numbers are needed to access the behaviour data
        frame_numbers = X_balanced[:, -1].astype(int) # Access the last column of the design matrix
        X_balanced = X_balanced[:, :-1] # Remove the last column of the design matrix
        
        # Compute the accuracy for each fold
        for fold, (train_index, test_index) in enumerate(group_kfold.split(X_balanced, y_balanced, groups_balanced)):
            print(f"Training fold: {fold}")
            X_train, X_test = X_balanced[train_index], X_balanced[test_index]
            y_train, y_test = y_balanced[train_index], y_balanced[test_index]
            if not check_there_are_two_classes(y_train): continue
            down_count_ans = count_class_labels(y_train)
            print(f"Class distribution for fold: {fold}. 0: {down_count_ans[0]}, 1: {down_count_ans[1]}")
            model = LogisticRegression(
                penalty='l2',
                dual=False, 
                tol=0.0001, 
                C=1.0, 
                fit_intercept=True, 
                intercept_scaling=1,  
                class_weight="balanced", 
                random_state=1337, 
                solver='lbfgs', 
                max_iter=10000, 
                verbose=0, 
                warm_start=False, 
                n_jobs=None, 
                l1_ratio=None
            ).fit(X_train, y_train)
            y_pred = model.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            fold_accuracies.append(accuracy)
            plot_the_train_test_behavioural_split(video_df, frame_numbers, train_index, test_index, dir, session_name, fold, accuracy)
            x_ks, y_ks, s_ks, h_ks = plot_class_distributions_and_compute_kolmogorov_Smirnov(frame_numbers, video_df, y_train, dir, session_name, fold, accuracy)
            ks_decoding_comparison[session_name][fold] = {"x": x_ks, "y": y_ks, "speed": s_ks, "hdir": h_ks, "accuracy": accuracy}
            coefs[session_name][fold] = model.coef_[0]
            
        # Compute the mean accuracy for the session
        whole_homing[session_name] = np.mean(fold_accuracies)
        print(f"Mean accuracy for session {session_name}: {whole_homing[session_name]}")
        
    return whole_homing, ks_decoding_comparison, coefs

# --- Plotting functions for the model ---

def plot_class_distributions_and_compute_kolmogorov_Smirnov(frame_numbers, video_df, y_train, dir, session_name, fold, accuracy):
           
    # Get the behavioural distributions of the classes for the training dataset to plot histograms to check they overlap
    good_indicies = np.where(y_train == 1)[0]
    bad_indicies = np.where(y_train == 0)[0]
    good_frames = frame_numbers[good_indicies]
    bad_frames = frame_numbers[bad_indicies]
    good_x_positions = video_df["mouse_x_position"][good_frames]
    bad_x_positions = video_df["mouse_x_position"][bad_frames]
    good_y_positions = video_df["mouse_y_position"][good_frames]
    bad_y_positions = video_df["mouse_y_position"][bad_frames]
    good_speed = video_df["speed"][good_frames]
    bad_speed = video_df["speed"][bad_frames]
    good_hdir = video_df["hdir"][good_frames]
    bad_hdir = video_df["hdir"][bad_frames]
    
    # Compute the Kolmogorov-Smirnov statistic to see if the distributions are different
    x_stat, x_ks, _ = are_the_distributions_different(good_x_positions, bad_x_positions)
    y_stat, y_ks, _ = are_the_distributions_different(good_y_positions, bad_y_positions)
    speed_stat, s_ks, _ = are_the_distributions_different(good_speed, bad_speed)
    hdir_stat, h_ks, _ = are_the_distributions_different(good_hdir, bad_hdir)
    
    # Plot the train class distributions
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    
    axes[0].hist(good_x_positions, bins=30, alpha=0.5, label='Good')
    axes[0].hist(bad_x_positions, bins=30, alpha=0.5, label='Bad')
    axes[0].set_title(f"X Position Distribution - Distributions Different: {x_stat}")
    axes[0].set_ylabel("Frequency")
    axes[0].legend()
    
    axes[1].hist(good_y_positions, bins=30, alpha=0.5, label='Good')
    axes[1].hist(bad_y_positions, bins=30, alpha=0.5, label='Bad')
    axes[1].set_title(f"Y Position Distribution - Distributions Different: {y_stat}")
    axes[1].set_ylabel("Frequency")
    axes[1].legend()
    
    axes[2].hist(good_speed, bins=30, alpha=0.5, label='Good')
    axes[2].hist(bad_speed, bins=30, alpha=0.5, label='Bad')
    axes[2].set_title(f"Speed Distribution - Are Distributions Different: {speed_stat}")
    axes[2].set_ylabel("Frequency")
    axes[2].legend()
    
    axes[3].hist(good_hdir, bins=30, alpha=0.5, label='Good')
    axes[3].hist(bad_hdir, bins=30, alpha=0.5, label='Bad')
    axes[3].set_title(f"Hdir Distribution - Are Distributions Different: {hdir_stat}")
    axes[3].set_ylabel("Frequency")
    axes[3].legend()
    plt.suptitle(f"Behavioural Data class break down training distributions for Session {session_name}, Fold {fold}, Accuracy: {accuracy}")
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(os.path.join(dir, f"class_distributions_behaviour_{session_name}_fold_{fold}.png"))
    plt.close()
    
    
    return x_ks, y_ks, s_ks, h_ks

def plot_the_train_test_behavioural_split(video_df, frame_numbers, train_index, test_index, save_dir, session_name, fold, accuracy):
    
    # Extract the frame numbers for the training and testing dataset 
    train_indices = frame_numbers[train_index]
    test_indices = frame_numbers[test_index]
    
    # Extract the behaviour data for the training and testing dataset so we can plot historgrams of the train and test data
    # to make sure they overlap within the folds
    x_position_train, x_position_test = video_df["mouse_x_position"][train_indices], video_df["mouse_x_position"][test_indices]
    y_position_train, y_position_test = video_df["mouse_y_position"][train_indices], video_df["mouse_y_position"][test_indices]
    speed_train, speed_test = video_df["speed"][train_indices], video_df["speed"][test_indices]
    hdir_train, hdir_test = video_df["hdir"][train_indices], video_df["hdir"][test_indices]
    
    # Plot the test-train distribution
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    
    axes[0].hist(x_position_train, bins=30, alpha=0.5, label='Train')
    axes[0].hist(x_position_test, bins=30, alpha=0.5, label='Test')
    x_stat, _, _ = are_the_distributions_different(x_position_train, x_position_test)
    axes[0].set_title(f"X Position Distribution - Are Distributions Different: {x_stat}")
    axes[0].set_ylabel("Frequency")
    axes[0].legend()
    
    axes[1].hist(y_position_train, bins=30, alpha=0.5, label='Train')
    axes[1].hist(y_position_test, bins=30, alpha=0.5, label='Test')
    y_stat, _, _ = are_the_distributions_different(y_position_train, y_position_test)
    axes[1].set_title(f"Y Position Distribution - Are Distributions Different: {y_stat}")
    axes[1].set_ylabel("Frequency")
    axes[1].legend()
    
    axes[2].hist(speed_train, bins=30, alpha=0.5, label='Train')
    axes[2].hist(speed_test, bins=30, alpha=0.5, label='Test')
    speed_stat, _, _ = are_the_distributions_different(speed_train, speed_test)
    axes[2].set_title(f"Speed Distribution - Are Distributions Different: {speed_stat}")
    axes[2].set_ylabel("Frequency")
    axes[2].legend()
    
    axes[3].hist(hdir_train, bins=30, alpha=0.5, label='Train')
    axes[3].hist(hdir_test, bins=30, alpha=0.5, label='Test')
    hdir_stat, _, _ = are_the_distributions_different(hdir_train, hdir_test)
    axes[3].set_title(f"Hdir Distribution - Are Distributions Different: {hdir_stat}")
    axes[3].set_ylabel("Frequency")
    axes[3].legend()

    plt.suptitle(f"Behavioural Data Distribution for Session {session_name}, Fold {fold}, Accuracy: {accuracy}")
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(os.path.join(save_dir, f"test_train_distributions_behaviour{session_name}_fold_{fold}.png"))
    plt.close()
    
def plot_accuracy_across_sessions(data_type, mice_groups, plot_title: str):

    # Process data into a DataFrame for easier manipulation
    data = list(data_type.items())
    df = pd.DataFrame(data, columns=['session', 'accuracy'])

    # Extract mouse ID
    df['mouse'] = df['session'].str.extract(r'(?P<mouse>JAL\d+)')

    # Sort sessions for each mouse based on their predefined order in mice_groups
    def sort_sessions(mouse, session):
        if mouse in mice_groups:
            return mice_groups[mouse].index(session)
        return np.nan

    df['session_order'] = df.apply(lambda x: sort_sessions(x['mouse'], x['session']), axis=1)

    # Drop any sessions that don't align with the mouse groups
    sorted_df = df.dropna(subset=['session_order']).sort_values(by=['mouse', 'session_order'])

    # Assign session numbers within each mouse group
    sorted_df['session_num'] = sorted_df.groupby('mouse').cumcount() + 1

    # Average accuracy across all sessions
    df_avg = sorted_df.groupby('session_num', as_index=False)['accuracy'].mean()

    # Visualization
    fig, ax = plt.subplots(figsize=(12, 6))

    # Plot average line across all mice
    ax.plot(df_avg['session_num'], df_avg['accuracy'], marker='o', linestyle='-', color='black', label='Average Across Mice')

    # Plot individual mice
    for mouse in sorted_df['mouse'].unique():
        mouse_data = sorted_df[sorted_df['mouse'] == mouse]
        ax.plot(mouse_data['session_num'], mouse_data['accuracy'], marker='o', linestyle='--', alpha=0.5, label=mouse)

    ax.set_xticks(range(1, sorted_df['session_num'].max() + 1))
    ax.set_xticklabels(range(1, sorted_df['session_num'].max() + 1))
    ax.set_title(f"Decoding Accuracy Across Mice and Sessions - {plot_title}")
    ax.set_xlabel("Session Number")
    ax.set_ylabel("Accuracy")
    ax.legend(title='Mouse ID')
    plt.tight_layout()
    plt.show()

# Generate the design matrices for different analysis experiments. E.g: Random labels, Random Neural Windows, Across a Session, Before a homing

In [9]:
#random_times_200_frames_5s = produce_data(experiments_objects, session_names, "random_times_200_frames_5s", create_the_random_design_matrix)

In [10]:
#compute_entire_spatial_efficiency = produce_data(experiments_objects, session_names, "compute_entire_spatial_efficiency", create_the_design_matrix)

In [ ]:
compute_the_first_20_frames_before_homing = produce_data(experiments_objects, session_names, "compute_the_first_20_frames_before_homing", create_the_past_design_matrix)

# Load pre-computed data

In [9]:
# Real data
#compute_entire_spatial_efficiency = load(dir, "compute_entire_spatial_efficiency.pkl")
compute_the_first_20_frames_before_homing = load(dir, "compute_the_first_20_frames_before_homing.pkl")

# Random neural windows - design matrix
#random_times_200_frames_5s = load(dir, "random_times_200_frames_5s.pkl")

# Other variants 

In [13]:
#whole_homing = compute_accuracy_data(data_type=compute_entire_spatial_efficiency)
#plot_accuracy_across_sessions(whole_homing, mice_groups=mice_groups, plot_title="Across the entire homing")
#whole_homing_random_class = compute_accuracy_data(data_type= compute_entire_spatial_efficiency, random_labels=True)
#plot_accuracy_across_sessions(whole_homing_random_class, mice_groups=mice_groups, plot_title="Random labels")
#random_neural_activity_windows = compute_accuracy_data(data_type=random_times_200_frames_5s, random_labels= False)
#plot_accuracy_across_sessions(random_neural_activity_windows, mice_groups=mice_groups, plot_title="Random neural windows")

# Now select the 500ms before the homing

In [ ]:
_500ms_before_homing, _500ms_before_ks_decoding_comparison, coefs = compute_accuracy_data(data_type=compute_the_first_20_frames_before_homing, data_type_name = "500ms_before_accuracy", random_labels= False)

Running logistic regression for session: JAL6_flip3_18mar
Class distribution before down sampling. 0: 460, 1: 1640
The length of balanced indcies 920
The length of homing_ids 2100
Class distribution for the entire session after down sampling. 0: 460, 1: 460
Training fold: 0
Class distribution for fold: 0. 0: 220, 1: 240
Training fold: 1
Class distribution for fold: 1. 0: 240, 1: 220
Mean accuracy for session JAL6_flip3_18mar: 0.6054347826086957
Running logistic regression for session: JAL6_flip4_21mar
Class distribution before down sampling. 0: 500, 1: 1780
The length of balanced indcies 1000
The length of homing_ids 2280
Class distribution for the entire session after down sampling. 0: 500, 1: 500
Training fold: 0
Class distribution for fold: 0. 0: 240, 1: 259
Training fold: 1
Class distribution for fold: 1. 0: 260, 1: 241
Mean accuracy for session JAL6_flip4_21mar: 0.5540262161048644
Running logistic regression for session: JAL6_flip5_25mar
Class distribution before down sampling. 0:

In [27]:
plot_accuracy_across_sessions(_500ms_before_homing, mice_groups=mice_groups, plot_title="500ms before homing")

# Plot the KS vs decoding accuracy analysis

In [ ]:
# Extract average, max KS stats, and accuracies
def extract_ks_and_accuracy(data):
    avg_ks_values = []
    max_ks_values = []
    accuracies = []

    for session, trials in data.items():
        for trial, metrics in trials.items():
            if all(k in metrics for k in ['x', 'y', 'speed', 'hdir', 'accuracy']):
                ks_values = [metrics['x'], metrics['y'], metrics['speed'], metrics['hdir']]
                avg_ks_values.append(np.mean(ks_values))
                max_ks_values.append(np.max(ks_values))
                accuracies.append(metrics['accuracy'])

    return np.array(avg_ks_values), np.array(max_ks_values), np.array(accuracies)

# Perform the correlation analysis
def analyze_correlation(ks_values, accuracies):
    correlation = np.corrcoef(ks_values, accuracies)[0, 1]
    return correlation

# Extract data
avg_ks_values, max_ks_values, accuracies = extract_ks_and_accuracy(_500ms_before_ks_decoding_comparison)

# Check for sufficient data
if len(avg_ks_values) > 0 and len(max_ks_values) > 0 and len(accuracies) > 0:
    # Correlations
    avg_correlation = analyze_correlation(avg_ks_values, accuracies)
    max_correlation = analyze_correlation(max_ks_values, accuracies)

    print(f"Correlation between average KS statistic and decoding accuracy: {avg_correlation}")
    print(f"Correlation between max KS statistic and decoding accuracy: {max_correlation}")

    # Plot correlations across all sessions and folds
    plt.figure()
    plt.scatter(avg_ks_values, accuracies, alpha=0.7, label='Average KS')
    plt.scatter(max_ks_values, accuracies, alpha=0.7, label='Max KS', color='r')
    plt.xlabel('KS Statistic')
    plt.title(f"Max and Average KS Statistic vs Decoding Accuracy (Average r={avg_correlation:.2f}, Max r={max_correlation:.2f})")
    plt.legend()
    plt.show()

    # Plot individual KS variables against decoding accuracy
    ks_variables = ['x', 'y', 'speed', 'hdir']
    fig, axs = plt.subplots(2, 2, figsize=(10, 10))
    axs = axs.ravel()

    for i, var in enumerate(ks_variables):
        ks_values = []
        accuracies = []
        for session, trials in _500ms_before_ks_decoding_comparison.items():
            for trial, metrics in trials.items():
                if var in metrics and 'accuracy' in metrics:
                    ks_values.append(metrics[var])
                    accuracies.append(metrics['accuracy'])

        correlation = analyze_correlation(ks_values, accuracies)
        axs[i].scatter(ks_values, accuracies, alpha=0.7)
        axs[i].set_title(f'{var} vs Accuracy (r={correlation:.2f})')
        axs[i].set_xlabel("KS Statistic")
        axs[i].set_ylabel('Accuracy')

    plt.tight_layout()
    plt.show()
else:
    print("Insufficient data for analysis.")

# --------------------------------------------- Appendix ---------------------------------------------

# Controls

In [ ]:
control_for_500ms_before_homing(experiments_objects, session_names, dir)

# Plot and compute additional metrics to see how the model can be improved

In [ ]:
def plot_session_metrics(whole_homing_metrics):
    """
    Plots accuracy, F1 score, precision, and recall across sessions.
    
    Args:
        whole_homing_metrics (dict): Metrics for each session.
    """
    # Convert to DataFrame
    data = []
    for session, metrics in whole_homing_metrics.items():
        metrics["session"] = session
        data.append(metrics)
    df = pd.DataFrame(data)

    # Extract mouse ID and session number
    df['mouse'] = df['session'].str.extract(r'(?P<mouse>JAL\d+)')
    df['session_num'] = df.groupby('mouse').cumcount() + 1

    # Plot metrics
    metrics = ['accuracy', 'f1_score', 'precision', 'recall']
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.flatten()

    for i, metric in enumerate(metrics):
        ax = axes[i]
        df_avg = df.groupby('session_num', as_index=False)[metric].mean()
        ax.plot(df_avg['session_num'], df_avg[metric], marker='o', linestyle='-', color='black', label='Average Across Mice')
        
        for mouse in df['mouse'].unique():
            mouse_data = df[df['mouse'] == mouse]
            ax.plot(mouse_data['session_num'], mouse_data[metric], marker='o', linestyle='--', alpha=0.5, label=mouse)

        ax.set_xticks(range(1, df['session_num'].max() + 1))
        ax.set_xticklabels(range(1, df['session_num'].max() + 1))
        ax.set_title(f"{metric.capitalize()} Across Sessions")
        ax.set_xlabel("Session Number")
        ax.set_ylabel(metric.capitalize())
        ax.legend(title='Mouse ID')

    plt.tight_layout()
    plt.show()
def compute_metrics_data(data_type, random_labels=False):
    """
    Enhances the model to compute accuracy, F1 score, precision, and recall.
    Stores the metrics per session.
    
    Args:
        data_type (dict): Dictionary containing the data for each session.
        random_labels (bool): If True, shuffle the labels randomly.

    Returns:
        dict: Contains accuracy, F1 score, precision, and recall for each session.
    """
    # Initialize storage for metrics
    whole_homing_metrics = defaultdict(dict)
    random_state = 1337
    
    for session_name, data in data_type.items():
        print("The session name is: ", session_name)
        design_matrix = data["design_matrix"]
        classes_extended = np.asarray(data["classes_extended"])
        
        # Randomize labels if requested
        if random_labels:
            np.random.shuffle(classes_extended)
        
        count_ans = count_class_labels(classes_extended)
        print(f"Class distribution before down sampling. 0: {count_ans[0]}, 1: {count_ans[1]}")
        if not check_there_is_enough_data(count_ans, session_name):
            continue
        
        homing_ids = np.asarray([item for sublist in data["homing_ids"] for item in sublist])
        X_balanced, y_balanced, groups_balanced = downsample_larger_class(
            design_matrix=design_matrix, classes_extended=classes_extended,
            random_state=random_state, homing_ids=homing_ids
        )
        count_of_downsampled_classes = count_class_labels(y_balanced)
        print(f"Class distribution after down sampling: 0: {count_of_downsampled_classes[0]}, 1: {count_of_downsampled_classes[1]}")
        
        # Perform GroupKFold
        group_kfold = GroupKFold(n_splits=2)
        accuracies, f1_scores, precisions, recalls = [], [], [], []
        
        for fold, (train_index, test_index) in enumerate(group_kfold.split(X_balanced, y_balanced, groups_balanced)):
            print(f"Training fold: {fold}")
            X_train, X_test = X_balanced[train_index], X_balanced[test_index]
            y_train, y_test = y_balanced[train_index], y_balanced[test_index]
            if not check_there_are_two_classes(y_train):
                continue
            
            model = LogisticRegression(
                penalty='l2',
                class_weight="balanced",
                random_state=random_state,
                solver='lbfgs',
                max_iter=10000
            ).fit(X_train, y_train)
            
            # Predictions and metrics
            y_pred = model.predict(X_test)
            accuracies.append(accuracy_score(y_test, y_pred))
            f1_scores.append(f1_score(y_test, y_pred))
            precisions.append(precision_score(y_test, y_pred))
            recalls.append(recall_score(y_test, y_pred))
        
        # Store metrics
        whole_homing_metrics[session_name]["accuracy"] = np.mean(accuracies)
        whole_homing_metrics[session_name]["f1_score"] = np.mean(f1_scores)
        whole_homing_metrics[session_name]["precision"] = np.mean(precisions)
        whole_homing_metrics[session_name]["recall"] = np.mean(recalls)
        
        print(f"Metrics for session {session_name}:")
        print(f"  Accuracy: {whole_homing_metrics[session_name]['accuracy']:.4f}")
        print(f"  F1 Score: {whole_homing_metrics[session_name]['f1_score']:.4f}")
        print(f"  Precision: {whole_homing_metrics[session_name]['precision']:.4f}")
        print(f"  Recall: {whole_homing_metrics[session_name]['recall']:.4f}")

    return whole_homing_metrics

_500ms_before_homing_extra = compute_metrics_data(data_type=compute_the_first_20_frames_before_homing, random_labels=False)
plot_session_metrics(_500ms_before_homing_extra)